# 08 — Đọc chuỗi thời gian bằng ADF, KPSS và ACF/PACF

Phân tích chính dùng lịch sử giá đến **29/11/2023**, ngày kết thúc Train. Lịch sử này gồm 4.214 mức giá và 4.213 log return. Kết quả trên toàn bộ bản dữ liệu được lưu riêng để mô tả.

Đọc diễn giải từng khái niệm và cảnh báo cách hiểu sai ở [EDA và đặc trưng](../docs/EDA_VA_DAC_TRUNG.md).

## Chuẩn bị môi trường

Trên máy cá nhân, cài `pip install -r requirements.txt` trước. Các lệnh mặc định dùng snapshot được lưu trong dự án; không tự thay dữ liệu bằng lần tải mạng mới.

In [ ]:
# Tìm thư mục gốc khi chạy từ repo, từ notebooks/ hoặc trên Google Colab.
from pathlib import Path
import sys, subprocess, json, os
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/fpt-stock-trends-prediction')]
ROOT = next((p for p in candidates if (p/'src/train.py').exists()), None)
if ROOT is None and 'google.colab' in sys.modules:
    subprocess.run(['git','clone','https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git','/content/fpt-stock-trends-prediction'], check=True)
    ROOT = Path('/content/fpt-stock-trends-prediction')
if ROOT is None:
    raise FileNotFoundError('Hãy mở notebook trong repository đã tải về.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'src'))
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
import pandas as pd
from IPython.display import display, Image
print('Thư mục làm việc:', ROOT)


## Hai kiểm định đặt hai giả thuyết khác nhau

“Dừng” không có nghĩa giá đứng yên; nó nói về độ ổn định của đặc điểm thống kê theo thời gian.

| Kiểm định | Giả thuyết gốc trong code | Nếu p < 0.05 |
|---|---|---|
| ADF | Có nghiệm đơn vị | Có bằng chứng bác bỏ nghiệm đơn vị |
| KPSS | Dừng quanh một mức trung bình hằng | Có bằng chứng bác bỏ dừng mức |

Trên log return, ADF p ≈ **5,63 × 10⁻¹⁷** và KPSS p ≈ **0,03382**. Ở ngưỡng 5%, ADF bác bỏ nghiệm đơn vị và KPSS bác bỏ dừng mức. Hai kết quả trả lời hai giả thuyết trong bảng.

**ACF** hỏi mức độ chuỗi giống chính nó ở mỗi độ trễ. **PACF** xem phần liên hệ còn lại sau khi tính đến các độ trễ ngắn hơn. Hình vẽ 40 độ trễ và dải tin cậy xấp xỉ 95%; cột vượt dải cần được xem xét, không bảo đảm dự báo đúng.

Ô dưới lưu bảng JSON/CSV và hình. KPSS có thể trả giới hạn bảng tra (ví dụ 0.01 kèm cảnh báo rằng p thật nhỏ hơn); cần đọc cả trường warnings.

In [ ]:
from time_series_analysis import run_analysis
result=run_analysis()
display(pd.read_csv(ROOT/'reports/stationarity_tests.csv'))
display(Image(filename=str(ROOT/'figures/12_acf_pacf.png')))

## Phân rã STL để quan sát, không đưa vào đặc trưng

STL tách **log(close)** thành `xu thế + thành phần lặp + phần dư`. Xu thế là chuyển động chậm, phần dư là những gì chưa được hai phần trước giải thích.

Code đặt chu kỳ **5 phiên giao dịch** và `robust=True` để bớt bị điểm cực đoan chi phối. Đây là chu kỳ quy ước để xem hình, không chứng minh có mùa vụ 5 phiên; 5 phiên cũng không luôn đúng một tuần lịch do ngày nghỉ.

Hình chỉ dùng lịch sử Train. Phép làm trơn của phân rã có thể dùng cả điểm trước/sau bên trong đoạn đó, nên không chép trực tiếp cột xu thế vào bảng dự báo rồi xem như đặc trưng chỉ nhìn quá khứ.

In [ ]:
display(Image(filename=str(ROOT/'figures/13_time_series_decomposition.png')))